# 01 — Exploración del dataset

Dataset: `pkmandke/Human-Posture-Dataset`  
Paper: H. Kale et al. (2018). IEEE IACC. DOI: 10.1109/IADCC.2018.8692143

**Objetivo:** entender la distribución, calidad y separabilidad de las 3 clases que usaremos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DATA_PATH = Path("../data/raw/posture_dataset.csv")
PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(exist_ok=True)

## 1. Carga y vista general

In [ ]:
df = pd.read_csv(DATA_PATH, index_col=0)
print(f"Shape: {df.shape}")
print(f"Columnas: {list(df.columns)}")
df.head()

In [ ]:
print(df.dtypes)
print(f"\nValores nulos: {df.isnull().sum().sum()}")
print(f"Valores inf: {np.isinf(df.select_dtypes('number').values).sum()}")

## 2. Distribución de clases (dataset original — 6 clases)

In [ ]:
LABEL_NAMES = {
    0.0: "sleeping",
    1.0: "standing",
    2.0: "sitting",
    3.0: "running",
    4.0: "forward_bending",
    5.0: "backward_bending",
}
df["posture"] = df["label"].map(LABEL_NAMES)
counts = df["posture"].value_counts()
print(counts)

plt.figure(figsize=(9, 4))
counts.plot(kind="bar", color="steelblue")
plt.title("Distribución de clases — dataset completo (6 clases)")
plt.ylabel("Muestras")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / "dist_6clases.png", dpi=100)
plt.show()

## 3. Filtrado a 3 clases del sistema SitRight

| Dataset | SitRight |
|---|---|
| sitting (2) | `adequate` |
| forward_bending (4) | `forward_slouch` |
| backward_bending (5) | `excessive_recline` |

Se descartan: sleeping, standing, running.

In [ ]:
KEEP = {2.0: "adequate", 4.0: "forward_slouch", 5.0: "excessive_recline"}
df3 = df[df["label"].isin(KEEP.keys())].copy()
df3["class"] = df3["label"].map(KEEP)

print(f"Shape filtrado: {df3.shape}")
print(df3["class"].value_counts())

plt.figure(figsize=(6, 4))
df3["class"].value_counts().plot(kind="bar", color=["#2ecc71", "#e74c3c", "#3498db"])
plt.title("Distribución de clases — 3 clases SitRight")
plt.ylabel("Muestras")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / "dist_3clases.png", dpi=100)
plt.show()

## 4. Distribuciones del sensor dorsal (chest) por clase

In [ ]:
features = ["Ax1", "Ay1", "Az1"]
classes = ["adequate", "forward_slouch", "excessive_recline"]
colors = {"adequate": "#2ecc71", "forward_slouch": "#e74c3c", "excessive_recline": "#3498db"}

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, feat in zip(axes, features):
    for cls in classes:
        data = df3[df3["class"] == cls][feat]
        ax.hist(data, bins=60, alpha=0.6, label=cls, color=colors[cls])
    ax.set_title(feat)
    ax.set_xlabel("Aceleración (g)")
    ax.legend(fontsize=8)
plt.suptitle("Distribuciones del sensor dorsal por clase", fontsize=13)
plt.tight_layout()
plt.savefig(PROCESSED_DIR / "histogramas_dorsal.png", dpi=100)
plt.show()

## 5. Pair plot — separabilidad visual

In [ ]:
sample = df3.groupby("class").sample(n=500, random_state=42)
g = sns.pairplot(sample[["Ax1", "Ay1", "Az1", "class"]], hue="class",
                 palette=colors, diag_kind="hist", plot_kws={"alpha": 0.4})
g.fig.suptitle("Pair plot — sensor dorsal (muestra 500 por clase)", y=1.02)
plt.savefig(PROCESSED_DIR / "pairplot_dorsal.png", dpi=100, bbox_inches="tight")
plt.show()

## 6. Estadísticas descriptivas por clase

In [ ]:
df3.groupby("class")[["Ax1", "Ay1", "Az1"]].describe().round(4)

## Conclusión

- Dataset sin valores nulos ni infinitos.
- Las 3 clases son visualmente separables en Az1 (eje vertical).
- `forward_slouch` y `excessive_recline` tienen distribuciones distintas en Ay1.
- Balance de clases aceptable — se usará `class_weight='balanced'` en el RF de todas formas.
- **Siguiente paso:** `02-preprocessing.ipynb`